# Train the Khmer OCR recognizer

Works unmodified on **Colab**, **Kaggle**, and a **local machine** -- on **CPU, GPU, or TPU**:
- Colab: mounts Google Drive at `/content/drive/My Drive/tuna-ocr` and checkpoints there.
- Kaggle: checkpoints to `/kaggle/working/tuna-ocr` (persisted as notebook output).
- Local: checkpoints to `recognizer/checkpoints/` in the repo.

The accelerator (TPU / GPU / CPU) is auto-detected -- no notebook changes needed either
way. **To actually get one**, select it as the runtime/accelerator in Colab
(Runtime > Change runtime type > GPU or TPU) or Kaggle (Settings > Accelerator >
GPU or TPU) *before* running this notebook; both platforms ship `torch_xla`
preinstalled on their TPU runtimes, so no extra install step is needed here. Detection
covers both TPU generations -- the legacy XRT env vars *and* the PJRT ones every
current runtime uses -- so section 1 prints the device it actually resolved
(`torch device: cuda` / `xla:0` / `cpu`); check that line before starting the run
rather than inferring it later from step timings.

On GPU, batch size is auto-probed to fit the available VRAM (OOM-probing auto-tune);
on TPU and CPU that probe is skipped -- XLA compiles lazily and never raises a
catchable Python OOM -- and `TrainConfig.batch_size` is used exactly as configured, so
set it yourself if you hit a TPU HBM limit.

This pulls a **real, at-scale dataset** by default (all of `deepcopy_khmer_text_recognition`
/ `darayut_scene_text` / `sokheng_synthetic_v1`, plus 100k rows of
`chanrith_ocr_image_line` -- see section 2), roughly ~9GB on disk -- this is a
production-scale training run, not the small notebook-scale loop in
`notebooks/train_diagnose_eval.ipynb`. Use that other notebook first if you just want
to iterate quickly on model/training-loop changes.

Training logs every 100 steps and pushes a checkpoint to the `Panhapich/tuna-ocr`
Hugging Face repo (created private) every 10,000 steps.

**Before running:** add an `HF_TOKEN` secret (Colab: key icon in the left sidebar;
Kaggle: Add-ons > Secrets; local: `export HF_TOKEN=hf_...`).

**TPU caveats** (GPU is the better-tested path here; TPU is supported but not tuned):
- PyTorch/XLA's support for the CTC loss op has historically been inconsistent across
  versions -- if training errors out or looks unusually slow on TPU, check whether
  `torch.nn.functional.ctc_loss` is silently falling back to a CPU path before assuming
  it's a bug in this repo.
- Batches here are variable-shaped by design (lines are bucketed by width, so the
  flattened chunk count and encoder length differ batch to batch). XLA compiles one
  graph per distinct shape, so expect a few hundred recompiles early in the run before
  the compilation cache covers the common shapes and step time settles.

**Platform settings to check first:**
- **Kaggle**: internet access is off by default -- Settings (right sidebar) > Internet >
  On. Without it, both `pip install` and the data-pull cell (which streams from the
  Hugging Face Hub) will fail. Also set Settings > Accelerator to GPU/TPU if you want
  one. Kaggle sessions have a disk budget too (check Settings) -- this notebook's
  ~9GB default pull should fit comfortably, but reduce `SAMPLES_PER_SOURCE` (section 2)
  if you're tight on space.
- **Colab**: Runtime > Change runtime type > pick GPU or TPU if you want one (default
  is CPU-only).


In [4]:
import os, subprocess, sys

def detect_environment():
    # Kaggle is checked first: some Kaggle kernels leak a stray COLAB_GPU/
    # COLAB_RELEASE_TAG env var, which would otherwise misdetect as Colab and
    # crash trying to mount Google Drive. "/kaggle/working" existing is a much
    # harder signal to spoof than an env var, so it takes priority.
    if "KAGGLE_KERNEL_RUN_TYPE" in os.environ or os.path.isdir("/kaggle/working"):
        return "kaggle"
    if "COLAB_RELEASE_TAG" in os.environ or "COLAB_GPU" in os.environ or "COLAB_TPU_ADDR" in os.environ:
        return "colab"
    return "local"

ENV = detect_environment()
print("environment:", ENV)

REPO_URL = "https://github.com/Pich09/tuna-ocr.git"
REPO_DIR = "tuna-ocr"

def run_git(args):
    """Runs git and raises with git's ACTUAL stderr on failure. A bare
    CalledProcessError only reports "exit status 128", which is git's catch-all
    and says nothing about which of the many possible causes (existing
    directory, auth, network) actually happened."""
    r = subprocess.run(["git", *args], capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"git {' '.join(args)} failed ({r.returncode}):\n{r.stderr.strip()}")
    return r

# Three cases, in order. The middle one is the important fix: after a kernel
# restart the cwd resets to /content (or /kaggle/working), so "recognizer" is no
# longer visible even though a previous run already cloned the repo -- the old
# code then tried to clone again and git aborted with "destination path already
# exists and is not an empty directory" (exit 128).
if os.path.isdir("recognizer"):
    print("already inside the repo working dir")
elif os.path.isdir(REPO_DIR):
    os.chdir(REPO_DIR)
    print(f"found an existing clone, reusing it: {os.getcwd()}")
    try:  # best effort -- a dirty tree or offline runtime shouldn't block training
        run_git(["pull", "--ff-only"])
        print("pulled latest changes")
    except RuntimeError as e:
        print(f"skipping pull (not fatal): {e}")
else:
    run_git(["clone", REPO_URL, REPO_DIR])
    os.chdir(REPO_DIR)
    print(f"cloned into {os.getcwd()}")

sys.path.insert(0, os.getcwd())
print("working dir:", os.getcwd())


environment: colab
already inside the repo working dir
working dir: /content/tuna-ocr


In [5]:
# Colab and Kaggle both ship torch preinstalled and matched to their runtime (the CUDA
# driver on a GPU runtime, or the torch_xla/libtpu build on a TPU runtime) -- blindly
# `pip install torch` on top of that (e.g. via a plain `-r recognizer/requirements.txt`)
# can silently replace it with a build that doesn't match, which breaks GPU support and
# breaks TPU support even harder (torch_xla is pinned to one exact torch version).
# Install everything else normally, and only pip-install torch if it isn't importable
# at all (a bare local venv).
import importlib.util
from pathlib import Path

def strip_torch(req_path):
    lines = Path(req_path).read_text().splitlines()
    return [l for l in lines if not l.strip().lower().startswith("torch")]

torch_before = None
if importlib.util.find_spec("torch") is not None:
    import torch
    torch_before = torch.__version__

reqs = strip_torch("recognizer/requirements.txt") + strip_torch("real_data/requirements.txt")
Path("/tmp/_notebook_requirements.txt").write_text("\n".join(reqs) + "\n")
!pip install -q -r /tmp/_notebook_requirements.txt

if torch_before is None:
    print("torch not found -- installing (no preinstalled build to preserve here)")
    !pip install -q torch
else:
    # Excluding torch from the requirements file isn't a complete guarantee: any
    # dependency in it is free to pull a *different* torch in as its own dependency.
    # On a TPU runtime that's silently fatal -- torch_xla only loads against the exact
    # torch build it was compiled for, and the failure surfaces much later as an opaque
    # import/libtpu error, so check explicitly here rather than discovering it then.
    # importlib.metadata, not `torch.__version__`: torch is already imported in this
    # kernel, so its module object still reports the OLD version no matter what pip
    # just wrote to disk (and importlib.reload(torch) is not a safe way to find out).
    # The distribution metadata reflects what's actually installed now.
    from importlib.metadata import version as _pkg_version
    torch_after = _pkg_version("torch")
    if torch_after != torch_before:
        print(f"WARNING: pip changed torch {torch_before} -> {torch_after} as a "
              f"transitive dependency. On a TPU runtime, restart the runtime and "
              f"`pip install torch=={torch_before}` before continuing, or torch_xla "
              f"will fail to load.")
    else:
        print(f"using preinstalled torch {torch_after} "
              f"(cuda available: {torch.cuda.is_available()}) -- not reinstalled")


using preinstalled torch 2.11.0+cu128 (cuda available: True) -- not reinstalled


In [6]:
import os

from recognizer import env_utils

checkpoint_root = env_utils.get_checkpoint_root(ENV)

# Token resolution, in order. Colab's own secret store (env_utils.get_hf_token) is
# tried first but is NOT reliable: it raises "Secrets can only be fetched when running
# from the Colab UI" whenever the notebook runs detached from the UI tab, which is
# exactly what happened on a long training run here. So fall back to an HF_TOKEN
# environment variable, then to a plain file, then to an interactive prompt --
# deliberately never hardcoded in this notebook, which is committed to a public git
# repo (GitHub's push protection rejects the commit outright, and HF's secret scanner
# auto-revokes any write-scoped token that lands in one).
#
# Easiest on Colab: run this in a scratch cell once per session, paste when prompted:
#     import os, getpass; os.environ["HF_TOKEN"] = getpass.getpass("HF token: ")
hf_token = None
try:
    hf_token = env_utils.get_hf_token(ENV)
except Exception as e:
    print(f"platform secret store unavailable ({type(e).__name__}), trying fallbacks...")
if not hf_token:
    hf_token = os.environ.get("HF_TOKEN")
if not hf_token:
    for candidate in ("/content/hf_token.txt", "/kaggle/working/hf_token.txt", "hf_token.txt"):
        if os.path.exists(candidate):
            hf_token = open(candidate).read().strip()
            print(f"read HF token from {candidate}")
            break
if not hf_token:
    import getpass
    hf_token = getpass.getpass("HF token (input hidden): ").strip()

# Resolve the accelerator NOW, before the multi-hour cells below, and print the exact
# torch device that training will use. detect_accelerator() covers both TPU
# generations (legacy XRT env vars and current PJRT ones) plus the /dev/accel* device
# nodes, so a modern Colab/Kaggle TPU runtime is recognised rather than falling through
# to CPU -- a fallback that is otherwise invisible until you notice steps taking 100x
# too long, hours in.
accelerator = env_utils.detect_accelerator()
device = env_utils.get_torch_device()

print("environment:      ", ENV)
print("checkpoint root:  ", checkpoint_root)
print("HF token loaded:  ", bool(hf_token))
print("accelerator:      ", env_utils.describe_accelerator())
print("torch device:     ", device)
if accelerator == "cpu":
    print("\n>>> No GPU/TPU detected. Colab: Runtime > Change runtime type. "
          "Kaggle: Settings > Accelerator. Do not start the training cell on CPU -- "
          "at this dataset's scale it will not finish.")


Mounted at /content/drive
platform secret store unavailable (RuntimeError), trying fallbacks...
environment:       colab
checkpoint root:   /content/drive/My Drive/tuna-ocr/checkpoints
HF token loaded:   True
accelerator:       cuda: Tesla T4 (15.6 GB)
torch device:      cuda


In [7]:
# Downloads Panhapich/khmer-sp-8k's SentencePiece model + khmer_segmentation.py
# wrapper (a bare .model file is not enough -- see recognizer/README.md).
from recognizer.tokenizer.fetch_tokenizer import fetch_tokenizer

fetch_tokenizer()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Fetched Panhapich/khmer-sp-8k -> /content/tuna-ocr/recognizer/tokenizer/assets: ['gazetteer.json', 'khmer_segmentation.py', 'khmer_sp.model', 'latin_exceptions.json', 'tokenizer_info.json']


PosixPath('/content/tuna-ocr/recognizer/tokenizer/assets')

## 2. Data

The full pull -> pack -> dedup pipeline below is expensive (real network transfer +
CPU-bound hashing, potentially a long time at this scale) -- it only needs to run
**once**. The first successful run pushes its result to a private Hugging Face
dataset repo (`real_data.config.HF_DATA_REPO_ID`); every later run (new session, new
notebook, different machine) checks that repo first and just downloads the prebuilt
`dedup.arrow` instead of repeating the pull/pack/dedup work from scratch.

`SAMPLES_PER_SOURCE` only matters the first time (before anything's been pushed to the
Hub). The defaults pull everything available from the three smaller sources, but cap
`chanrith_ocr_image_line` (12M+ rows, ~40GB in full) at 100k rows. Each source is
pulled, packed into a single Arrow file (`<source>.arrow`, image bytes stored exactly
as pulled -- no re-encoding/resizing), and its raw per-image files are deleted before
the next source starts, bounding peak disk usage to "one source's raw files + all
Arrow files packed so far."


In [8]:
import os, shutil, subprocess, sys
from pathlib import Path
from real_data.config import EXTERNAL_DATASETS, HF_DATA_REPO_ID, REAL_DATA_ROOT
from real_data import hf_push

# Per-source sample counts for the (one-time) pull from source. Pulls everything
# available from the smaller sources, but caps chanrith_ocr_image_line (12M+ rows) at
# 100k -- pulling it in full would be ~40GB, far more than a Kaggle/Colab session's
# disk budget can hold.
SAMPLES_PER_SOURCE = {
    "deepcopy_khmer_text_recognition": 136_117,
    "chanrith_ocr_image_line": 100_000,
    "darayut_scene_text": 102_500,
    "sokheng_synthetic_v1": 100_000,
}

# KMP_DUPLICATE_LIB_OK/OMP_NUM_THREADS: Colab/Kaggle commonly have more than one
# OpenMP runtime on the import path (numpy, PIL/imagehash, datasets' native deps each
# bundle their own libomp/libiomp5) -- loading two in one process is a well-known cause
# of an immediate SIGABRT with zero output, right at import time, before any of this
# script's own code runs. Setting these before spawning avoids that class of crash;
# harmless if it wasn't actually the cause.
SUBPROCESS_ENV = {**os.environ, "KMP_DUPLICATE_LIB_OK": "TRUE", "OMP_NUM_THREADS": "1"}

def run_checked(cmd):
    """Runs `cmd`, always printing its output, and raises with the actual captured
    stderr on failure -- a bare `subprocess.CalledProcessError` (or, worse, a `!shell`
    cell whose exit code isn't checked at all) hides exactly the text that explains
    *why* it died, which is the difference between a one-line fix and a guessing game."""
    result = subprocess.run(cmd, env=SUBPROCESS_ENV, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout)
    if result.returncode != 0:
        sys.stderr.write(result.stderr)
        raise RuntimeError(
            f"command failed (exit code {result.returncode}"
            f"{', likely killed by a signal -- see stderr above for the real cause' if result.returncode < 0 else ''}"
            f"): {' '.join(cmd)}"
        )
    return result

dedup_manifest = REAL_DATA_ROOT / "samples" / "dedup.arrow"
built_locally = False  # tracks whether THIS run built dedup_manifest from source
                        # (vs. it already being local, or downloaded from the Hub) --
                        # only push to the Hub in the first case (section 2b below).

if dedup_manifest.exists():
    print(f"{dedup_manifest} already present locally, skipping pull/download")
elif hf_push.dataset_exists_on_hub(HF_DATA_REPO_ID, token=hf_token):
    print(f"found a prebuilt dataset on the Hub ({HF_DATA_REPO_ID}) -- downloading "
          f"instead of re-pulling/re-deduplicating from source")
    hf_push.pull_dataset(dedup_manifest, token=hf_token, repo_id=HF_DATA_REPO_ID)
else:
    print(f"no prebuilt dataset found on {HF_DATA_REPO_ID} -- pulling + packing from "
          f"source (one-time cost; result gets pushed to the Hub in the next cell)")
    built_locally = True

    # Pull -> pack to Arrow -> delete raw, one source at a time (not all sources
    # pulled first, then packed): this bounds peak disk usage to "current source's
    # raw files + every Arrow file packed so far", instead of needing all 4 sources'
    # raw files on disk simultaneously.
    arrow_files = []
    for source in EXTERNAL_DATASETS:
        arrow_path = REAL_DATA_ROOT / "samples" / f"{source}.arrow"
        arrow_files.append(arrow_path)
        if arrow_path.exists():
            print(f"{source}: already packed, skipping")
            continue

        source_dir = REAL_DATA_ROOT / "samples" / source
        num_samples = SAMPLES_PER_SOURCE[source]
        if not (source_dir / "manifest.tsv").exists():
            print(f"{source}: pulling {num_samples} samples...")
            run_checked([sys.executable, "-m", "real_data.generate_external_chunks",
                         "--source", source, "--num-samples", str(num_samples)])

        print(f"{source}: packing to {arrow_path}...")
        run_checked([sys.executable, "-m", "real_data.pack_arrow",
                     "--source", source, "--delete-raw"])

    print(arrow_files)


found a prebuilt dataset on the Hub (Panhapich/tuna-ocr-data) -- downloading instead of re-pulling/re-deduplicating from source


dedup.arrow: reconstructing file:   0%|          |  0.00B / 3.95GB            

dedup.arrow: downloading bytes:           |  0.00B            

In [9]:
# 2b. Deduplicate + push to the Hub -- only runs if this session actually built the
# dataset from source above (built_locally == True); a no-op if dedup_manifest was
# already local or was just downloaded from the Hub.
if built_locally:
    # --near-dup-threshold 0 disables the O(n^2) near-dup pass -- REQUIRED at this
    # scale (hundreds of thousands of rows): the default pairwise comparison is
    # O(n^2) and would take an impractically long time (the nonzero default is only
    # tuned/safe for the notebook-scale hundreds-to-thousands range, e.g.
    # notebooks/train_diagnose_eval.ipynb).
    missing = [str(p) for p in arrow_files if not p.exists()]
    if missing:
        raise RuntimeError(
            "The following sources are missing their packed .arrow file -- re-run "
            "the pull cell above (in full, for all 4 sources) before deduplicating. "
            "This usually means the runtime restarted/reset between the pull and "
            "dedup cells (e.g. after a crash) and the previously-pulled data under "
            f"{REAL_DATA_ROOT} was lost:\n  " + "\n  ".join(missing)
        )

    run_checked([sys.executable, "-m", "real_data.deduplicate",
                 "--arrow-files", *[str(p) for p in arrow_files],
                 "--out", str(dedup_manifest),
                 "--near-dup-threshold", "0"])
    assert dedup_manifest.exists(), (
        f"{dedup_manifest} was not created -- check the pull cell above actually "
        f"populated {[str(p) for p in arrow_files]} before dedup ran."
    )
    print("dedup arrow file ready:", dedup_manifest)

    print(f"pushing prebuilt dataset to the Hub ({HF_DATA_REPO_ID}) so future runs "
          f"can skip straight to downloading it...")
    url = hf_push.push_dataset(dedup_manifest, token=hf_token, repo_id=HF_DATA_REPO_ID, private=True)
    print("pushed:", url)
else:
    print(f"{dedup_manifest} already ready (local or from the Hub) -- nothing to dedup/push")


/content/tuna-ocr/real_data/samples/dedup.arrow already ready (local or from the Hub) -- nothing to dedup/push


In [10]:
from pathlib import Path

from recognizer.config import ModelConfig, TrainConfig
from recognizer.train import run_training
from recognizer.hf_push import pull_latest_checkpoint

model_cfg = ModelConfig()
# Knobs worth touching from here rather than editing library code in Colab.
# LOG_EVERY: 100 gives NO output at all until step 100 completes, which is
#   indistinguishable from a hang on a cold start (XLA especially, where the first
#   steps pay graph compilation). Drop it to 1-10 for the first minutes of a new run
#   to confirm steps are actually advancing, then raise it back.
LOG_EVERY = 100
# NUM_WORKERS: 0 is the safe default *on CUDA* (see below). On a TPU or CPU runtime
#   there is no CUDA context to fork after, so 4 is safe there and takes the ~47ms/batch
#   of image-decode + tokenize off the critical path.
NUM_WORKERS = 0

train_cfg = TrainConfig(
    log_every=LOG_EVERY,
    num_workers=NUM_WORKERS,  # DataLoader workers fork *after* CUDA is initialized in a
                    # notebook kernel, which is unsafe and can deadlock silently (GPU
                    # pinned at 100% with zero real steps). 0 = safe synchronous loading.
    max_eval_samples=512,     # Val samples the periodic eval covers for CTC CER. Cheap:
                    # an argmax over an encoder pass, ~26ms/sample all-in.
    max_ar_eval_samples=64,   # ...but the AR greedy decode inside that eval emits ONE
                    # TOKEN PER FORWARD PASS in sequential mode. Measured on a real run:
                    # 512 AR decodes = 639s (10.7 min) every eval_every=500 steps, against
                    # 265s of actual training -- 71% of wall clock. 64 keeps the signal
                    # without owning the run; 0 = CTC CER only.
    sequential_ar_steps=20_000,  # ~10% of max_steps=200_000 (default). Trains the AR
                                 # decoder in plain sequential mode first (full teacher
                                 # forcing, unrestricted cross-attention) so it learns
                                 # real image-to-text alignment before switching to
                                 # blockwise mode -- see recognizer/README.md's
                                 # "Blockwise-AR decoding" section for why: training
                                 # blockwise from step 0 was observed to make the AR
                                 # decoder fit the uniform block-split approximation
                                 # instead of the image, capping its accuracy well
                                 # below CTC's on the same encoder.
)  # log_every=100, ckpt_every=10_000 by default

RUN_NAME = "v1"
CHECKPOINT_REPO_ID = "Panhapich/tuna-ocr"

# False (current setting): always start from scratch, ignoring any checkpoint that
# already exists locally or on the Hub -- the new run's step_*.pt files overwrite the
# old ones of the same name in CHECKPOINT_REPO_ID as it goes. Set True to instead
# continue from where a previous session left off: it prefers a local
# checkpoints/<RUN_NAME>/last.pt (same session, merely interrupted), then falls back
# to the highest-step checkpoint pushed to the Hub (fresh Colab/Kaggle session, whose
# local disk was wiped), then starts fresh if neither exists.
RESUME_FROM_CHECKPOINT = False

resume_path = None
if RESUME_FROM_CHECKPOINT:
    local_last = Path(checkpoint_root) / RUN_NAME / "last.pt"
    if local_last.exists():
        resume_path = local_last
        print(f"resuming from local checkpoint: {resume_path}")
    else:
        resume_path = pull_latest_checkpoint(Path(checkpoint_root) / RUN_NAME, token=hf_token,
                                             repo_id=CHECKPOINT_REPO_ID)
        if resume_path:
            print(f"no local checkpoint -- resuming from the latest one on the Hub: {resume_path}")
        else:
            print(f"no local or Hub checkpoint found for {CHECKPOINT_REPO_ID} -- starting a fresh run")
else:
    print("RESUME_FROM_CHECKPOINT is False -- starting from scratch and overwriting "
          f"existing checkpoints in {CHECKPOINT_REPO_ID} as the run progresses")

# The Panhapich/tuna-ocr HF repo is created private by default the first time
# a checkpoint is pushed (hub_private=True) -- set to False only if you've
# deliberately decided the checkpoint repo should be public.
model = run_training(
    model_cfg, train_cfg,
    dedup_manifest_path=dedup_manifest,
    checkpoint_root=checkpoint_root,
    run_name=RUN_NAME,
    push_to_hub=True,
    repo_id=CHECKPOINT_REPO_ID,
    hf_token=hf_token,
    hub_private=True,
    auto_batch_size=True,  # OOM-probing auto-tune. CUDA only -- on TPU/CPU the
                       # configured TrainConfig.batch_size is used as-is, since XLA
                       # compiles lazily and never raises a catchable Python OOM.
    resume_path=resume_path,
)


RESUME_FROM_CHECKPOINT is False -- starting from scratch and overwriting existing checkpoints in Panhapich/tuna-ocr as the run progresses
run_training: loading tokenizer...
run_training: loading samples from /content/tuna-ocr/real_data/samples/dedup.arrow...
run_training: loaded 428911 samples, shuffling/splitting...
run_training: building char vocab...
run_training: building model + moving to cuda...


2026-07-30 02:19:18 run 'v1': device=cuda, 420333 train / 8578 val samples, checkpoints -> /content/drive/My Drive/tuna-ocr/checkpoints/v1, log file -> /content/drive/My Drive/tuna-ocr/checkpoints/v1/train.log
2026-07-30 02:19:56 image widths ready in 38.2s (cache: /content/tuna-ocr/real_data/samples/widths_cache.json)
| 2026-07-30 02:19:58,614 | INFO | khmer-nltk | Loaded model from /usr/local/lib/python3.12/dist-packages/khmernltk/word_tokenize/sklearn_crf_ner_10000.sav |
INFO:khmer-nltk:Loaded model from /usr/local/lib/python3.12/dist-packages/khmernltk/word_tokenize/sklearn_crf_ner_10000.sav
2026-07-30 02:20:02 auto batch size: 64
2026-07-30 02:20:02 batch_size=64, ~6567 steps/epoch, max_steps=200000 (~30.5 epochs)
2026-07-30 02:20:09 AR decoder training curriculum: sequential mode for steps 0-20000 (plain teacher forcing, unrestricted cross-attention -- see modules/decoder.py), then blockwise for the rest.
/content/tuna-ocr/recognizer/data/dataset.py:57: UserWarning: [sokheng_synt

KeyboardInterrupt: 